# Spendly optimized model research notebook

This notebook tests the strongest leakage-safe changes **before any app integration**.
It reproduces a chronological evaluation on `combined_finance_raw.csv` and exports every
metric, prediction, fitted research artifact, and a JSON handoff block.

The notebook keeps three anomaly tracks separate:

- enhanced Isolation Forest: unsupervised and closest to the current project design;
- causal rules + Isolation Forest: deployable hybrid using only information available at transaction time;
- supervised Extra Trees diagnostic: an upper-bound experiment trained on controlled synthetic labels.

The supervised result must not be presented as real-world fraud performance. This dataset is synthetic,
and the labels encode generator patterns.

## How to run

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (the notebook also runs on CPU).
2. Run all cells.
3. When prompted, upload the exact `combined_finance_raw.csv` used in the previous notebook.
4. Download `spendly_optimized_output.zip` at the end.
5. Send back the text between `SPENDLY_OPTIMIZED_RESULT_START` and `SPENDLY_OPTIMIZED_RESULT_END`.

No application files or database records are changed by this notebook.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
    "tensorflow": "tensorflow", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "joblib": "joblib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check: PASS")

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import time
import warnings
import zipfile
from collections import defaultdict, deque
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, IsolationForest, RandomForestClassifier,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    confusion_matrix, mean_absolute_error, mean_squared_error,
    precision_recall_curve, precision_recall_fscore_support, r2_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
SEED = 42
LOOKBACK = 8
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
OUTPUT_DIR = Path("spendly_optimized_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print({"tensorflow": tf.__version__, "seed": SEED, "output_dir": str(OUTPUT_DIR.resolve())})

In [ ]:
requested_path = os.environ.get("SPENDLY_DATA_PATH", "combined_finance_raw.csv")
DATA_PATH = Path(requested_path)
if not DATA_PATH.exists():
    matches = list(Path.cwd().rglob("combined_finance_raw.csv"))
    if matches:
        DATA_PATH = matches[0]
    else:
        try:
            from google.colab import files
            uploaded = files.upload()
            DATA_PATH = Path(next(iter(uploaded)))
        except Exception as exc:
            raise FileNotFoundError(
                "Upload combined_finance_raw.csv or set SPENDLY_DATA_PATH."
            ) from exc

CSV = DATA_PATH
dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw["transaction_timestamp"] = pd.to_datetime(raw["transaction_timestamp"], errors="raise")
raw["amount"] = pd.to_numeric(raw["amount"], errors="raise")
required_columns = {
    "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
    "transaction_type", "merchant", "is_recurring", "user_monthly_income",
    "user_age", "user_employment_status", "user_income_day", "is_anomaly", "anomaly_type",
}
missing_columns = sorted(required_columns - set(raw.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert raw["transaction_id"].is_unique, "transaction_id must be unique"
assert raw["amount"].ge(0).all(), "Amounts must be non-negative"
dataset_summary = {
    "sha256": dataset_sha256,
    "rows": int(len(raw)),
    "users": int(raw["user_id"].nunique()),
    "start": str(raw["transaction_timestamp"].min()),
    "end": str(raw["transaction_timestamp"].max()),
    "synthetic_rows": int(raw.get("is_synthetic", pd.Series(False, index=raw.index)).fillna(False).astype(bool).sum()),
}
display(pd.Series(dataset_summary, name="value").to_frame())
print("Dataset contract and identity checks: PASS")

## 1. Forecasting improvements

The main change is to tell the model about **future-known calendar context**—for example,
whether rent day, utilities day, subscription day, savings day, or payday proximity falls in
the target week. Recurring values are estimated only from the preceding eight weeks.

Model choices follow official guidance for chronological splitting, robust Huber loss, and
non-negative long-tailed targets: [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html),
[TensorFlow Huber](https://www.tensorflow.org/api_docs/python/tf/keras/losses/huber), and
[scikit-learn Poisson regression example](https://scikit-learn.org/stable/auto_examples/linear_model/plot_poisson_regression_non_normal_loss.html).

In [ ]:
def bools(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    data = pd.read_csv(CSV, low_memory=False)
    data["transaction_timestamp"] = pd.to_datetime(data["transaction_timestamp"])
    data["amount"] = pd.to_numeric(data["amount"])
    data["is_recurring"] = bools(data["is_recurring"])
    data["transaction_type"] = data["transaction_type"].astype(str).str.lower()
    data["category"] = data["category"].astype(str).str.strip()
    profiles = (
        data[["user_id", "user_age", "user_monthly_income", "user_employment_status", "user_income_day"]]
        .drop_duplicates("user_id")
        .set_index("user_id")
    )
    return data, profiles


def build_weekly(data: pd.DataFrame, profiles: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    working = data.copy()
    working["period_start"] = working["transaction_timestamp"].dt.to_period("W-SUN").dt.start_time
    expense = working[working["transaction_type"].eq("expense")].copy()
    income = working[working["transaction_type"].eq("income")].copy()
    expense["recurring_amount"] = np.where(expense["is_recurring"], expense["amount"], 0.0)
    base = expense.groupby(["user_id", "period_start"]).agg(
        total_spending=("amount", "sum"),
        transaction_count=("transaction_id", "size"),
        average_transaction_amount=("amount", "mean"),
        recurring_expenses=("recurring_amount", "sum"),
    ).reset_index()
    category = expense.pivot_table(
        index=["user_id", "period_start"], columns="category", values="amount", aggfunc="sum", fill_value=0.0
    ).rename(columns=lambda value: "category_" + str(value).lower().replace(" ", "_")).reset_index()
    category_columns = sorted(c for c in category if c.startswith("category_"))
    income_agg = income.groupby(["user_id", "period_start"])["amount"].sum().rename("income").reset_index()
    grid = pd.MultiIndex.from_product(
        [sorted(data["user_id"].unique()), pd.date_range(working["period_start"].min(), working["period_start"].max(), freq="W-MON")],
        names=["user_id", "period_start"],
    ).to_frame(index=False)
    weekly = grid.merge(base, how="left", on=["user_id", "period_start"]).merge(
        category, how="left", on=["user_id", "period_start"]
    ).merge(income_agg, how="left", on=["user_id", "period_start"])
    numeric = ["total_spending", "transaction_count", "average_transaction_amount", "recurring_expenses", "income", *category_columns]
    weekly[numeric] = weekly[numeric].fillna(0.0)
    weekly = weekly.join(profiles, on="user_id")
    weekly = weekly.sort_values(["user_id", "period_start"]).reset_index(drop=True)
    grouped = weekly.groupby("user_id", group_keys=False)
    weekly["previous_period_spending"] = grouped["total_spending"].shift(1).fillna(0.0)
    weekly["rolling_average_4"] = grouped["total_spending"].transform(
        lambda values: values.shift(1).rolling(4, min_periods=1).mean()
    ).fillna(0.0)
    previous = weekly["previous_period_spending"]
    weekly["spending_growth_rate"] = np.where(
        previous > 0, (weekly["total_spending"] - previous) / previous, 0.0
    )
    weekly["spending_growth_rate"] = weekly["spending_growth_rate"].replace([np.inf, -np.inf], 0.0)
    features = [
        "total_spending", *category_columns, "transaction_count", "average_transaction_amount", "income",
        "recurring_expenses", "previous_period_spending", "rolling_average_4", "spending_growth_rate",
    ]
    return weekly, features


def day_in_week(period_start: pd.Timestamp, day: int) -> float:
    dates = pd.date_range(period_start, period_start + pd.Timedelta(days=6), freq="D")
    return float(any(d.day == day for d in dates))


def days_before_payday_in_week(period_start: pd.Timestamp, payday: int) -> float:
    dates = pd.date_range(period_start, period_start + pd.Timedelta(days=6), freq="D")
    return float(sum(((payday - int(d.day)) % 30) <= 3 for d in dates))


def nonzero_median(values: np.ndarray) -> np.ndarray:
    output = np.zeros(values.shape[0], dtype=float)
    for i, row in enumerate(values):
        positive = row[row > 0]
        output[i] = float(np.median(positive)) if len(positive) else 0.0
    return output


def build_sequences(weekly: pd.DataFrame, features: list[str]) -> dict[str, np.ndarray]:
    rows = []
    for user_id, group in weekly.groupby("user_id", sort=True):
        group = group.sort_values("period_start").reset_index(drop=True)
        values = group[features].to_numpy(float)
        for end in range(LOOKBACK - 1, len(group) - 1):
            target_idx = end + 1
            rows.append((
                str(user_id),
                values[end - LOOKBACK + 1:end + 1],
                float(group.loc[target_idx, "total_spending"]),
                pd.Timestamp(group.loc[target_idx, "period_start"]),
                float(group.loc[target_idx, "user_monthly_income"]),
                float(group.loc[target_idx, "user_age"]),
                str(group.loc[target_idx, "user_employment_status"]),
                int(group.loc[target_idx, "user_income_day"]),
            ))
    user = np.array([r[0] for r in rows])
    history = np.stack([r[1] for r in rows])
    target = np.array([r[2] for r in rows])
    dates = pd.to_datetime([r[3] for r in rows]).to_numpy()
    income = np.array([r[4] for r in rows])
    age = np.array([r[5] for r in rows])
    status = np.array([r[6] for r in rows])
    payday = np.array([r[7] for r in rows])

    total_idx = features.index("total_spending")
    recurring_categories = {
        "rent": features.index("category_rent"),
        "utilities": features.index("category_utilities"),
        "subscriptions": features.index("category_subscriptions"),
        "savings": features.index("category_savings"),
    }
    totals = history[:, :, total_idx]
    recurring_history = sum(history[:, :, idx] for idx in recurring_categories.values())
    discretionary = np.maximum(0.0, totals - recurring_history)
    target_context = []
    calendar_baseline = []
    for i, target_date in enumerate(pd.to_datetime(dates)):
        flags = {
            "rent": day_in_week(target_date, 2),
            "utilities": day_in_week(target_date, 10),
            "subscriptions": day_in_week(target_date, 15),
            "savings": day_in_week(target_date, 26),
        }
        recurring_estimate = 0.0
        for name, idx in recurring_categories.items():
            recurring_estimate += flags[name] * nonzero_median(history[i:i+1, :, idx])[0]
        discretionary_estimate = float(np.median(discretionary[i, -4:]))
        calendar_baseline.append(recurring_estimate + discretionary_estimate)
        month_angle = 2 * math.pi * (target_date.month - 1) / 12
        week_angle = 2 * math.pi * (target_date.isocalendar().week - 1) / 52
        target_context.append([
            income[i], age[i], payday[i],
            float(status[i] == "student"), float(status[i] == "employed"), float(status[i] == "self_employed"),
            flags["rent"], flags["utilities"], flags["subscriptions"], flags["savings"],
            days_before_payday_in_week(target_date, payday[i]),
            math.sin(month_angle), math.cos(month_angle), math.sin(week_angle), math.cos(week_angle),
            float(target_date.month), float(target_date.day),
            float(np.mean(totals[i, -4:])), float(np.median(totals[i, -4:])), float(np.std(totals[i, -4:])),
            float(np.mean(discretionary[i, -4:])), float(np.median(discretionary[i, -4:])),
            float(totals[i, -1] - totals[i, -4]),
            float(calendar_baseline[-1]),
        ])
    target_context = np.asarray(target_context, dtype=float)
    X = np.concatenate([history.reshape(len(history), -1), target_context], axis=1)
    return {
        "X": X, "history": history, "context": target_context,
        "y": target, "date": dates, "user": user, "income": income,
        "calendar_baseline": np.asarray(calendar_baseline),
    }


def forecast_metric_record(actual: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    pred = np.maximum(0.0, np.asarray(pred))
    ae = np.abs(actual - pred)
    return {
        "mae": float(ae.mean()),
        "rmse": float(mean_squared_error(actual, pred) ** 0.5),
        "wape": float(100 * ae.sum() / np.abs(actual).sum()),
        "r2": float(r2_score(actual, pred)),
        "bias": float(np.mean(pred - actual)),
    }


def candidate_specs() -> dict[str, tuple[object, str]]:
    return {
        "linear_context": (LinearRegression(n_jobs=-1), "raw"),
        "ridge_10_raw": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "raw"),
        "ridge_100_raw": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "raw"),
        "ridge_10_log": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "log"),
        "ridge_100_log": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "log"),
        "ridge_10_residual": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "residual"),
        "ridge_100_residual": (make_pipeline(StandardScaler(), Ridge(alpha=100.0)), "residual"),
        "ridge_10_income_ratio": (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), "ratio"),
        "hist_poisson": (HistGradientBoostingRegressor(loss="poisson", learning_rate=0.05, max_iter=250, max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=10.0, random_state=SEED), "raw"),
        "hist_squared": (HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.05, max_iter=250, max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=10.0, random_state=SEED), "raw"),
        "gbr_huber": (GradientBoostingRegressor(loss="huber", learning_rate=0.03, n_estimators=250, max_depth=2, min_samples_leaf=20, random_state=SEED), "raw"),
        "extra_trees": (ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.7, n_jobs=-1, random_state=SEED), "raw"),
    }


def fit_predict(spec: tuple[object, str], Xtr: np.ndarray, ytr: np.ndarray, btr: np.ndarray, itr: np.ndarray, Xva: np.ndarray, bva: np.ndarray, iva: np.ndarray) -> np.ndarray:
    estimator, transform = spec
    estimator = clone(estimator)
    if transform == "log":
        estimator.fit(Xtr, np.log1p(ytr))
        return np.expm1(estimator.predict(Xva))
    if transform == "residual":
        estimator.fit(Xtr, ytr - btr)
        return bva + estimator.predict(Xva)
    if transform == "ratio":
        estimator.fit(Xtr, ytr / np.maximum(itr, 1.0))
        return estimator.predict(Xva) * iva
    estimator.fit(Xtr, ytr)
    return estimator.predict(Xva)

In [ ]:
data, profiles = load_data()
weekly, forecast_features = build_weekly(data, profiles)
seq = build_sequences(weekly, forecast_features)
unique_target_dates = np.sort(np.unique(seq["date"]))
pretest_dates = unique_target_dates[:-3]
forecast_evaluation_dates = unique_target_dates[-3:]
fold_ranges = [(9, 11), (11, 13), (13, 15)]

forecast_split_table = pd.DataFrame([
    {"split": "model-selection window", "target_weeks": len(pretest_dates),
     "start": pd.Timestamp(pretest_dates.min()), "end": pd.Timestamp(pretest_dates.max())},
    {"split": "held-out evaluation", "target_weeks": len(forecast_evaluation_dates),
     "start": pd.Timestamp(forecast_evaluation_dates.min()), "end": pd.Timestamp(forecast_evaluation_dates.max())},
])
display(forecast_split_table)
assert pretest_dates.max() < forecast_evaluation_dates.min()
print({
    "samples": len(seq["y"]), "lookback_weeks": LOOKBACK,
    "history_features": len(forecast_features), "target_context_features": seq["context"].shape[1],
})
print("Strict chronological forecast separation: PASS")

In [ ]:
all_specs = candidate_specs()
candidate_names = [
    "gbr_huber", "hist_poisson", "hist_squared", "extra_trees",
    "ridge_100_residual", "linear_context",
]
cv_records = []
fitted_seconds = {}
for name in candidate_names:
    fold_records = []
    started = time.perf_counter()
    for train_count, validation_end in fold_ranges:
        train_mask = np.isin(seq["date"], pretest_dates[:train_count])
        validation_mask = np.isin(seq["date"], pretest_dates[train_count:validation_end])
        prediction = fit_predict(
            all_specs[name], seq["X"][train_mask], seq["y"][train_mask],
            seq["calendar_baseline"][train_mask], seq["income"][train_mask],
            seq["X"][validation_mask], seq["calendar_baseline"][validation_mask],
            seq["income"][validation_mask],
        )
        fold_records.append(forecast_metric_record(seq["y"][validation_mask], prediction))
    elapsed = time.perf_counter() - started
    fitted_seconds[name] = elapsed
    cv_records.append({
        "model": name,
        "validation_mae": float(np.mean([item["mae"] for item in fold_records])),
        "validation_wape": float(np.mean([item["wape"] for item in fold_records])),
        "validation_r2": float(np.mean([item["r2"] for item in fold_records])),
        "seconds": elapsed,
    })

calendar_records = []
for train_count, validation_end in fold_ranges:
    validation_mask = np.isin(seq["date"], pretest_dates[train_count:validation_end])
    calendar_records.append(forecast_metric_record(
        seq["y"][validation_mask], seq["calendar_baseline"][validation_mask]
    ))
cv_records.append({
    "model": "calendar_baseline",
    "validation_mae": float(np.mean([item["mae"] for item in calendar_records])),
    "validation_wape": float(np.mean([item["wape"] for item in calendar_records])),
    "validation_r2": float(np.mean([item["r2"] for item in calendar_records])),
    "seconds": 0.0,
})
forecast_validation = pd.DataFrame(cv_records).sort_values("validation_mae").reset_index(drop=True)
locked_forecast_order = forecast_validation["model"].tolist()
display(forecast_validation.round(3))
print("Validation winner (fixed before evaluation):", locked_forecast_order[0])

In [ ]:
forecast_train_mask = np.isin(seq["date"], pretest_dates)
forecast_test_mask = np.isin(seq["date"], forecast_evaluation_dates)
forecast_test_records = []
prediction_frames = []
fitted_forecast_models = {}

for name in locked_forecast_order:
    if name == "calendar_baseline":
        prediction = seq["calendar_baseline"][forecast_test_mask]
    else:
        model, transform = all_specs[name]
        fitted = clone(model)
        x_train = seq["X"][forecast_train_mask]
        y_train = seq["y"][forecast_train_mask]
        baseline_train = seq["calendar_baseline"][forecast_train_mask]
        income_train = seq["income"][forecast_train_mask]
        if transform == "residual":
            fitted.fit(x_train, y_train - baseline_train)
            prediction = seq["calendar_baseline"][forecast_test_mask] + fitted.predict(seq["X"][forecast_test_mask])
        elif transform == "log":
            fitted.fit(x_train, np.log1p(y_train))
            prediction = np.expm1(fitted.predict(seq["X"][forecast_test_mask]))
        elif transform == "ratio":
            fitted.fit(x_train, y_train / np.maximum(income_train, 1.0))
            prediction = fitted.predict(seq["X"][forecast_test_mask]) * seq["income"][forecast_test_mask]
        else:
            fitted.fit(x_train, y_train)
            prediction = fitted.predict(seq["X"][forecast_test_mask])
        fitted_forecast_models[name] = fitted
    prediction = np.maximum(0.0, np.asarray(prediction))
    forecast_test_records.append({"model": name, **forecast_metric_record(seq["y"][forecast_test_mask], prediction)})
    prediction_frames.append(pd.DataFrame({
        "model": name, "user_id": seq["user"][forecast_test_mask],
        "target_week": pd.to_datetime(seq["date"][forecast_test_mask]),
        "actual": seq["y"][forecast_test_mask], "predicted": prediction,
    }))

forecast_evaluation = pd.DataFrame(forecast_test_records).sort_values("mae").reset_index(drop=True)
forecast_predictions = pd.concat(prediction_frames, ignore_index=True)
display(forecast_evaluation.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=forecast_evaluation, x="mae", y="model", ax=axes[0], color="#2563eb")
axes[0].set_title("Held-out forecast MAE (lower is better)")
best_name = forecast_evaluation.iloc[0]["model"]
best_points = forecast_predictions[forecast_predictions["model"].eq(best_name)]
axes[1].scatter(best_points["actual"], best_points["predicted"], alpha=0.35, s=18)
limit = max(best_points["actual"].max(), best_points["predicted"].max())
axes[1].plot([0, limit], [0, limit], "--", color="black")
axes[1].set(title=f"{best_name}: actual vs predicted", xlabel="Actual KES", ylabel="Predicted KES")
plt.tight_layout()
plt.show()

### Context-aware LSTM

The LSTM receives the eight historical weeks and the target-week calendar context through
separate inputs. Raw, log, and calendar-residual targets are compared on validation only;
only the validation winner is evaluated on the final three weeks.

In [ ]:
lstm_train_dates = unique_target_dates[:12]
lstm_validation_dates = unique_target_dates[12:15]
lstm_train_mask = np.isin(seq["date"], lstm_train_dates)
lstm_validation_mask = np.isin(seq["date"], lstm_validation_dates)

history_scaler = StandardScaler()
context_scaler = StandardScaler()
history_scaler.fit(seq["history"][lstm_train_mask].reshape(-1, len(forecast_features)))
context_scaler.fit(seq["context"][lstm_train_mask])

def scaled_lstm_inputs(mask):
    history = seq["history"][mask]
    scaled_history = history_scaler.transform(history.reshape(-1, len(forecast_features))).reshape(history.shape)
    return scaled_history.astype("float32"), context_scaler.transform(seq["context"][mask]).astype("float32")

def build_context_lstm():
    history_input = tf.keras.Input(shape=(LOOKBACK, len(forecast_features)), name="history")
    context_input = tf.keras.Input(shape=(seq["context"].shape[1],), name="target_context")
    x = tf.keras.layers.LSTM(32, return_sequences=True, dropout=0.10)(history_input)
    x = tf.keras.layers.LSTM(16, dropout=0.10)(x)
    c = tf.keras.layers.Dense(24, activation="swish")(context_input)
    c = tf.keras.layers.Dropout(0.10)(c)
    joined = tf.keras.layers.Concatenate()([x, c])
    joined = tf.keras.layers.Dense(32, activation="swish")(joined)
    joined = tf.keras.layers.Dropout(0.10)(joined)
    output = tf.keras.layers.Dense(1)(joined)
    model = tf.keras.Model([history_input, context_input], output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0),
    )
    return model

def transform_target(kind, y, baseline, center=None, scale=None):
    values = np.log1p(y) if kind == "log" else y - baseline if kind == "residual" else y
    if center is None:
        center = float(np.mean(values))
        scale = float(np.std(values)) or 1.0
    return ((values - center) / scale).astype("float32"), center, scale

def inverse_target(kind, prediction, baseline, center, scale):
    values = np.asarray(prediction).reshape(-1) * scale + center
    return np.expm1(values) if kind == "log" else baseline + values if kind == "residual" else values

xh_train, xc_train = scaled_lstm_inputs(lstm_train_mask)
xh_validation, xc_validation = scaled_lstm_inputs(lstm_validation_mask)
xh_test, xc_test = scaled_lstm_inputs(forecast_test_mask)

In [ ]:
lstm_validation_records = []
lstm_candidates = {}
for target_kind in ["raw", "log", "residual"]:
    y_scaled, center, scale = transform_target(
        target_kind, seq["y"][lstm_train_mask], seq["calendar_baseline"][lstm_train_mask]
    )
    y_validation_scaled, _, _ = transform_target(
        target_kind, seq["y"][lstm_validation_mask],
        seq["calendar_baseline"][lstm_validation_mask], center, scale
    )
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = build_context_lstm()
    started = time.perf_counter()
    history = model.fit(
        [xh_train, xc_train], y_scaled,
        validation_data=([xh_validation, xc_validation], y_validation_scaled),
        epochs=160, batch_size=128, shuffle=False, verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=18, min_delta=1e-4, restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", patience=7, factor=0.5, min_lr=1e-5
            ),
        ],
    )
    prediction = inverse_target(
        target_kind, model.predict([xh_validation, xc_validation], verbose=0),
        seq["calendar_baseline"][lstm_validation_mask], center, scale,
    )
    lstm_validation_records.append({
        "target": target_kind,
        **forecast_metric_record(seq["y"][lstm_validation_mask], prediction),
        "epochs": len(history.history["loss"]), "seconds": time.perf_counter() - started,
    })
    lstm_candidates[target_kind] = (model, center, scale)

lstm_validation = pd.DataFrame(lstm_validation_records).sort_values("mae").reset_index(drop=True)
selected_lstm_target = str(lstm_validation.iloc[0]["target"])
context_lstm, lstm_target_center, lstm_target_scale = lstm_candidates[selected_lstm_target]
lstm_test_prediction = inverse_target(
    selected_lstm_target, context_lstm.predict([xh_test, xc_test], verbose=0),
    seq["calendar_baseline"][forecast_test_mask], lstm_target_center, lstm_target_scale,
)
lstm_test_record = {
    "model": "context_lstm_" + selected_lstm_target,
    **forecast_metric_record(seq["y"][forecast_test_mask], lstm_test_prediction),
}
display(lstm_validation.round(3))
display(pd.Series(lstm_test_record, name="held-out evaluation").to_frame())
forecast_evaluation = pd.concat(
    [forecast_evaluation, pd.DataFrame([lstm_test_record])], ignore_index=True
).sort_values("mae").reset_index(drop=True)
forecast_predictions = pd.concat([forecast_predictions, pd.DataFrame({
    "model": lstm_test_record["model"], "user_id": seq["user"][forecast_test_mask],
    "target_week": pd.to_datetime(seq["date"][forecast_test_mask]),
    "actual": seq["y"][forecast_test_mask], "predicted": np.maximum(0.0, lstm_test_prediction),
})], ignore_index=True)
print("Combined held-out forecast ranking")
display(forecast_evaluation.round(3))

## 2. Anomaly improvements

Every behavioural feature is causal: only earlier transactions for the same user are visible.
The expanded feature set adds income-normalized amounts, category-relative values, one-hour and
one-day frequency, same-day accumulation, recurring baselines, and exact duplicate timing.

The enhanced Isolation Forest follows the official
[scikit-learn Isolation Forest example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_isolation_forest.html).
Controlled labels select validation thresholds and evaluate results; they are never Isolation Forest inputs.

In [ ]:
def safe_ratio(numerator: float, denominator: float, neutral: float = 0.0) -> float:
    return float(numerator / denominator) if denominator and np.isfinite(denominator) else neutral


def build_past_only_features(raw: pd.DataFrame) -> pd.DataFrame:
    expense = raw[raw["transaction_type"].astype(str).str.lower().eq("expense")].copy()
    expense["transaction_timestamp"] = pd.to_datetime(expense["transaction_timestamp"])
    expense["amount"] = pd.to_numeric(expense["amount"], errors="coerce")
    expense["is_recurring"] = bools(expense["is_recurring"])
    expense["is_anomaly_target"] = bools(expense["is_anomaly"])
    expense = expense.sort_values(["user_id", "transaction_timestamp", "transaction_id"]).reset_index(drop=True)

    engineered_rows: list[dict] = []
    for user_id, group in expense.groupby("user_id", sort=False):
        q1h: deque[tuple[pd.Timestamp, float]] = deque()
        q24h: deque[tuple[pd.Timestamp, float]] = deque()
        q7d: deque[tuple[pd.Timestamp, float]] = deque()
        sum24h = 0.0
        sum7d = 0.0
        user_amounts: list[float] = []
        category_amounts: dict[str, list[float]] = defaultdict(list)
        recurring_amounts: dict[str, list[float]] = defaultdict(list)
        category_counts: dict[str, int] = defaultdict(int)
        last_exact: dict[tuple[str, str, float], pd.Timestamp] = {}
        last_merchant_category: dict[tuple[str, str], pd.Timestamp] = {}
        last_timestamp: pd.Timestamp | None = None
        day_key = None
        day_count = 0
        day_sum = 0.0

        for row in group.itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            amount = float(row.amount)
            category = str(row.category)
            merchant = str(row.merchant)
            income = float(row.user_monthly_income)
            current_day = timestamp.date()
            if day_key != current_day:
                day_key = current_day
                day_count = 0
                day_sum = 0.0

            cut1h = timestamp - pd.Timedelta(hours=1)
            cut24h = timestamp - pd.Timedelta(days=1)
            cut7d = timestamp - pd.Timedelta(days=7)
            while q1h and q1h[0][0] < cut1h:
                q1h.popleft()
            while q24h and q24h[0][0] < cut24h:
                sum24h -= q24h.popleft()[1]
            while q7d and q7d[0][0] < cut7d:
                sum7d -= q7d.popleft()[1]

            user_mean = float(np.mean(user_amounts)) if user_amounts else np.nan
            user_median = float(np.median(user_amounts)) if user_amounts else np.nan
            cat_values = category_amounts[category]
            category_mean = float(np.mean(cat_values)) if cat_values else np.nan
            category_median = float(np.median(cat_values)) if cat_values else np.nan
            recurring_values = recurring_amounts[category]
            recurring_median = float(np.median(recurring_values)) if recurring_values else np.nan
            exact_key = (merchant, category, round(amount, 2))
            merchant_key = (merchant, category)
            exact_age = (
                (timestamp - last_exact[exact_key]).total_seconds() / 60.0
                if exact_key in last_exact else np.nan
            )
            merchant_age = (
                (timestamp - last_merchant_category[merchant_key]).total_seconds() / 60.0
                if merchant_key in last_merchant_category else np.nan
            )
            past_count = len(user_amounts)

            engineered_rows.append({
                "transaction_id": row.transaction_id,
                "log_amount": np.log1p(max(0.0, amount)),
                "amount_to_income": safe_ratio(amount, income),
                "amount_to_user_mean": safe_ratio(amount, user_mean, 1.0),
                "amount_to_user_median": safe_ratio(amount, user_median, 1.0),
                "amount_to_category_mean": safe_ratio(amount, category_mean, 1.0),
                "amount_to_category_median": safe_ratio(amount, category_median, 1.0),
                "recurring_amount_ratio": safe_ratio(amount, recurring_median, 1.0),
                "past_user_count": past_count,
                "category_proportion": safe_ratio(category_counts[category], past_count),
                "previous_1h_count": len(q1h),
                "previous_24h_count": len(q24h),
                "previous_7d_count": len(q7d),
                "previous_24h_sum_to_income": safe_ratio(sum24h, income),
                "previous_7d_sum_to_income": safe_ratio(sum7d, income),
                "same_day_count_before": day_count,
                "same_day_sum_to_income": safe_ratio(day_sum, income),
                "time_since_previous_hours": (
                    (timestamp - last_timestamp).total_seconds() / 3600.0 if last_timestamp is not None else np.nan
                ),
                "same_exact_age_minutes": exact_age,
                "same_merchant_category_age_minutes": merchant_age,
                "is_recurring_feature": int(bool(row.is_recurring)),
                "is_weekend": int(timestamp.dayofweek >= 5),
                "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
                "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
            })

            q1h.append((timestamp, amount))
            q24h.append((timestamp, amount))
            q7d.append((timestamp, amount))
            sum24h += amount
            sum7d += amount
            user_amounts.append(amount)
            category_amounts[category].append(amount)
            category_counts[category] += 1
            if bool(row.is_recurring):
                recurring_amounts[category].append(amount)
            last_exact[exact_key] = timestamp
            last_merchant_category[merchant_key] = timestamp
            last_timestamp = timestamp
            day_count += 1
            day_sum += amount

    engineered = pd.DataFrame(engineered_rows)
    return expense.merge(engineered, on="transaction_id", how="left", validate="one_to_one")


def split_chronologically(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_times = np.sort(frame["transaction_timestamp"].unique())
    train_count = max(1, int(np.floor(len(unique_times) * 0.70)))
    train_end = unique_times[train_count - 1]
    train = frame[frame["transaction_timestamp"] <= train_end].copy()
    held = frame[frame["transaction_timestamp"] > train_end].copy()
    held_times = np.sort(held["transaction_timestamp"].unique())
    val_count = max(1, int(np.floor(len(held_times) * 0.20)))
    val_end = held_times[val_count - 1]
    validation = held[held["transaction_timestamp"] <= val_end].copy()
    evaluation = held[held["transaction_timestamp"] > val_end].copy()
    return train, validation, evaluation


def anomaly_metric_record(actual: np.ndarray, predicted: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="binary", zero_division=0
    )
    return {
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "false_positive_rate": float(fp / (fp + tn)), "tp": int(tp), "fp": int(fp),
        "tn": int(tn), "fn": int(fn), "alerts": int(predicted.sum()),
    }


def best_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds):
        threshold = float(np.max(scores) + 1)
    else:
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, anomaly_metric_record(labels, scores >= threshold)


def rules(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = frame["previous_1h_count"].ge(config["burst_count"])
    recurring = (
        frame["is_recurring_feature"].eq(1)
        & frame["category"].eq("Rent")
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    flags = np.column_stack([duplicate, burst, recurring, weekend, random_shock, extreme, rare]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0])
    return flags.dot(weights), flags


def percentile_against(reference: np.ndarray, values: np.ndarray) -> np.ndarray:
    ordered = np.sort(np.asarray(reference, dtype=float))
    return np.searchsorted(ordered, values, side="right") / max(1, len(ordered))

In [ ]:
anomaly_frame = build_past_only_features(raw)
anomaly_train, anomaly_validation, anomaly_evaluation = split_chronologically(anomaly_frame)
anomaly_split_table = pd.DataFrame([
    {"split": name, "rows": len(part), "anomalies": int(part["is_anomaly_target"].sum()),
     "start": part["transaction_timestamp"].min(), "end": part["transaction_timestamp"].max()}
    for name, part in [
        ("train", anomaly_train), ("validation", anomaly_validation), ("evaluation", anomaly_evaluation)
    ]
])
display(anomaly_split_table)
assert anomaly_train["transaction_timestamp"].max() < anomaly_validation["transaction_timestamp"].min()
assert anomaly_validation["transaction_timestamp"].max() < anomaly_evaluation["transaction_timestamp"].min()

anomaly_features = [
    "log_amount", "amount_to_income", "amount_to_user_mean", "amount_to_user_median",
    "amount_to_category_mean", "amount_to_category_median", "recurring_amount_ratio",
    "category_proportion", "previous_1h_count", "previous_24h_count", "previous_7d_count",
    "previous_24h_sum_to_income", "previous_7d_sum_to_income", "same_day_count_before",
    "same_day_sum_to_income", "time_since_previous_hours", "same_exact_age_minutes",
    "same_merchant_category_age_minutes", "is_recurring_feature", "is_weekend", "hour_sin", "hour_cos",
]
for part in [anomaly_train, anomaly_validation, anomaly_evaluation]:
    part[anomaly_features] = part[anomaly_features].replace([np.inf, -np.inf], np.nan)
anomaly_fill_values = anomaly_train[anomaly_features].median().fillna(0.0)
anomaly_scaler = RobustScaler(quantile_range=(10, 90))
anomaly_x_train = anomaly_scaler.fit_transform(anomaly_train[anomaly_features].fillna(anomaly_fill_values))
anomaly_x_validation = anomaly_scaler.transform(anomaly_validation[anomaly_features].fillna(anomaly_fill_values))
anomaly_x_evaluation = anomaly_scaler.transform(anomaly_evaluation[anomaly_features].fillna(anomaly_fill_values))
anomaly_y_train = anomaly_train["is_anomaly_target"].to_numpy(bool)
anomaly_y_validation = anomaly_validation["is_anomaly_target"].to_numpy(bool)
anomaly_y_evaluation = anomaly_evaluation["is_anomaly_target"].to_numpy(bool)
print("Strict chronological anomaly separation and training-only scaling: PASS")

In [ ]:
if_candidates = [
    {"name": "if_256_all", "max_samples": 256, "max_features": 1.0},
    {"name": "if_1024_all", "max_samples": 1024, "max_features": 1.0},
    {"name": "if_2048_75pct", "max_samples": 2048, "max_features": 0.75},
    {"name": "if_4096_all", "max_samples": 4096, "max_features": 1.0},
]
if_validation_rows = []
if_models = {}
for candidate in if_candidates:
    model = IsolationForest(
        n_estimators=400, max_samples=candidate["max_samples"],
        max_features=candidate["max_features"], contamination="auto",
        random_state=SEED, n_jobs=-1,
    ).fit(anomaly_x_train)
    validation_score = -model.score_samples(anomaly_x_validation)
    threshold, record = best_threshold(anomaly_y_validation, validation_score)
    if_validation_rows.append({**candidate, "threshold": threshold, **record})
    if_models[candidate["name"]] = (model, threshold)
if_validation = pd.DataFrame(if_validation_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
best_if_name = str(if_validation.iloc[0]["name"])
best_if_model, best_if_threshold = if_models[best_if_name]
best_if_train_scores = -best_if_model.score_samples(anomaly_x_train)
best_if_validation_scores = -best_if_model.score_samples(anomaly_x_validation)
best_if_evaluation_scores = -best_if_model.score_samples(anomaly_x_evaluation)
display(if_validation.round(4))

In [ ]:
rule_configs = []
for burst_count in [1, 2, 3, 4]:
    for extreme_ratio in [0.30, 0.40, 0.50]:
        rule_configs.append({
            "duplicate_minutes": 5, "burst_count": burst_count, "recurring_ratio": 1.35,
            "weekend_ratio": 0.06, "shock_ratio": 0.15, "extreme_ratio": extreme_ratio,
            "rare_proportion": 0.025, "rare_amount_ratio": 0.15,
        })
rule_validation_rows = []
for config_index, config in enumerate(rule_configs):
    validation_score, _ = rules(anomaly_validation, config)
    for threshold in [1.0, 2.0, 3.0, 4.0, 5.0]:
        rule_validation_rows.append({
            "config_index": config_index, "threshold": threshold,
            **anomaly_metric_record(anomaly_y_validation, validation_score >= threshold),
        })
rule_validation = pd.DataFrame(rule_validation_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
best_rule_row = rule_validation.iloc[0]
best_rule_config = rule_configs[int(best_rule_row["config_index"])]
best_rule_threshold = float(best_rule_row["threshold"])
rule_validation_score, _ = rules(anomaly_validation, best_rule_config)
rule_evaluation_score, rule_evaluation_flags = rules(anomaly_evaluation, best_rule_config)

if_validation_percentile = percentile_against(best_if_train_scores, best_if_validation_scores)
if_evaluation_percentile = percentile_against(best_if_train_scores, best_if_evaluation_scores)
hybrid_validation_rows = []
for if_weight in [0.5, 1.0, 1.5, 2.0]:
    hybrid_validation_score = rule_validation_score + if_weight * if_validation_percentile
    threshold, record = best_threshold(anomaly_y_validation, hybrid_validation_score)
    hybrid_validation_rows.append({"if_weight": if_weight, "threshold": threshold, **record})
hybrid_validation = pd.DataFrame(hybrid_validation_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
best_hybrid = hybrid_validation.iloc[0]
hybrid_evaluation_score = (
    rule_evaluation_score + float(best_hybrid["if_weight"]) * if_evaluation_percentile
)
print("Best causal rule configuration")
display(pd.Series({**best_rule_config, **best_rule_row.to_dict()}, name="value").to_frame())
print("Hybrid validation")
display(hybrid_validation.round(4))

In [ ]:
supervised_candidates = {
    "extra_trees_supervised": ExtraTreesClassifier(
        n_estimators=500, min_samples_leaf=2, class_weight="balanced",
        random_state=SEED, n_jobs=-1,
    ),
    "random_forest_supervised": RandomForestClassifier(
        n_estimators=500, max_depth=14, min_samples_leaf=2,
        class_weight="balanced_subsample", random_state=SEED, n_jobs=-1,
    ),
}
supervised_validation_rows = []
supervised_models = {}
for name, model in supervised_candidates.items():
    model.fit(anomaly_x_train, anomaly_y_train)
    validation_score = model.predict_proba(anomaly_x_validation)[:, 1]
    threshold, record = best_threshold(anomaly_y_validation, validation_score)
    supervised_validation_rows.append({"model": name, "threshold": threshold, **record})
    supervised_models[name] = (model, threshold)
supervised_validation = pd.DataFrame(supervised_validation_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
best_supervised_name = str(supervised_validation.iloc[0]["model"])
best_supervised_model, best_supervised_threshold = supervised_models[best_supervised_name]
display(supervised_validation.round(4))
print("CAUTION: supervised results are synthetic-label diagnostics, not real-world fraud estimates.")

In [ ]:
anomaly_predictions = {
    best_if_name: best_if_evaluation_scores >= best_if_threshold,
    "causal_rules": rule_evaluation_score >= best_rule_threshold,
    "rules_plus_isolation_forest": hybrid_evaluation_score >= float(best_hybrid["threshold"]),
    best_supervised_name: (
        best_supervised_model.predict_proba(anomaly_x_evaluation)[:, 1] >= best_supervised_threshold
    ),
}
anomaly_evaluation_records = [
    {"model": name, **anomaly_metric_record(anomaly_y_evaluation, prediction)}
    for name, prediction in anomaly_predictions.items()
]
anomaly_evaluation_metrics = pd.DataFrame(anomaly_evaluation_records).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
display(anomaly_evaluation_metrics.round(4))

anomaly_prediction_rows = []
scenario_rows = []
for name, prediction in anomaly_predictions.items():
    part = anomaly_evaluation[[
        "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
        "merchant", "anomaly_type", "is_anomaly_target",
    ]].copy()
    part["model"] = name
    part["predicted_anomaly"] = prediction
    anomaly_prediction_rows.append(part)
    scenario = (
        part[part["is_anomaly_target"]]
        .groupby("anomaly_type")["predicted_anomaly"]
        .agg(["size", "sum"])
        .reset_index()
        .rename(columns={"size": "labelled", "sum": "detected"})
    )
    scenario["recall"] = scenario["detected"] / scenario["labelled"]
    scenario["model"] = name
    scenario_rows.append(scenario)
anomaly_evaluation_predictions = pd.concat(anomaly_prediction_rows, ignore_index=True)
anomaly_scenario_metrics = pd.concat(scenario_rows, ignore_index=True)
display(anomaly_scenario_metrics.pivot(index="anomaly_type", columns="model", values="recall").round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=anomaly_evaluation_metrics, x="f1", y="model", ax=axes[0], color="#16a34a")
axes[0].set_xlim(0, 1)
axes[0].set_title("Held-out anomaly F1")
best_deployable = "rules_plus_isolation_forest"
deployable_prediction = anomaly_predictions[best_deployable]
matrix = confusion_matrix(anomaly_y_evaluation, deployable_prediction, labels=[False, True])
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[1].set_title("Deployable hybrid confusion matrix")
axes[1].set(xlabel="Predicted", ylabel="Controlled label")
plt.tight_layout()
plt.show()

## 3. Save the research bundle

These are notebook-only research artifacts. Review the results before changing the app.
The JSON block is deliberately compact so it can be pasted back into the Codex task.

In [ ]:
forecast_validation.to_csv(OUTPUT_DIR / "forecast_validation_metrics.csv", index=False)
forecast_evaluation.to_csv(OUTPUT_DIR / "forecast_evaluation_metrics.csv", index=False)
forecast_predictions.to_csv(OUTPUT_DIR / "forecast_evaluation_predictions.csv", index=False)
forecast_split_table.to_csv(OUTPUT_DIR / "forecast_split_summary.csv", index=False)
lstm_validation.to_csv(OUTPUT_DIR / "lstm_validation_metrics.csv", index=False)

if_validation.to_csv(OUTPUT_DIR / "isolation_forest_validation_metrics.csv", index=False)
rule_validation.to_csv(OUTPUT_DIR / "rule_validation_metrics.csv", index=False)
hybrid_validation.to_csv(OUTPUT_DIR / "hybrid_validation_metrics.csv", index=False)
supervised_validation.to_csv(OUTPUT_DIR / "supervised_validation_metrics.csv", index=False)
anomaly_evaluation_metrics.to_csv(OUTPUT_DIR / "anomaly_evaluation_metrics.csv", index=False)
anomaly_evaluation_predictions.to_csv(OUTPUT_DIR / "anomaly_evaluation_predictions.csv", index=False)
anomaly_scenario_metrics.to_csv(OUTPUT_DIR / "anomaly_scenario_metrics.csv", index=False)
anomaly_split_table.to_csv(OUTPUT_DIR / "anomaly_split_summary.csv", index=False)

if "gbr_huber" in fitted_forecast_models:
    joblib.dump(fitted_forecast_models["gbr_huber"], OUTPUT_DIR / "gbr_huber_research.joblib")
context_lstm.save(OUTPUT_DIR / "context_lstm_research.keras")
joblib.dump(history_scaler, OUTPUT_DIR / "lstm_history_scaler.joblib")
joblib.dump(context_scaler, OUTPUT_DIR / "lstm_context_scaler.joblib")
joblib.dump(best_if_model, OUTPUT_DIR / "enhanced_isolation_forest.joblib")
joblib.dump(anomaly_scaler, OUTPUT_DIR / "anomaly_scaler.joblib")
joblib.dump(best_supervised_model, OUTPUT_DIR / "synthetic_label_extra_trees.joblib")
(OUTPUT_DIR / "feature_schema.json").write_text(json.dumps({
    "forecast_history_features": forecast_features,
    "forecast_context_feature_count": int(seq["context"].shape[1]),
    "forecast_lookback": LOOKBACK,
    "anomaly_features": anomaly_features,
    "anomaly_fill_values": anomaly_fill_values.to_dict(),
    "best_rule_config": best_rule_config,
    "best_rule_threshold": best_rule_threshold,
}, indent=2, default=float), encoding="utf-8")

best_forecast = forecast_evaluation.iloc[0].to_dict()
best_unsupervised = anomaly_evaluation_metrics[
    anomaly_evaluation_metrics["model"].eq(best_if_name)
].iloc[0].to_dict()
best_hybrid_record = anomaly_evaluation_metrics[
    anomaly_evaluation_metrics["model"].eq("rules_plus_isolation_forest")
].iloc[0].to_dict()
best_supervised_record = anomaly_evaluation_metrics[
    anomaly_evaluation_metrics["model"].eq(best_supervised_name)
].iloc[0].to_dict()
final_payload = {
    "notebook": "Spending_Model_Optimized_Research",
    "dataset": dataset_summary,
    "forecast_validation_winner": forecast_validation.iloc[0].to_dict(),
    "forecast_best_held_out": best_forecast,
    "forecast_all_held_out": forecast_evaluation.round(6).to_dict(orient="records"),
    "selected_lstm_target": selected_lstm_target,
    "anomaly_best_unsupervised_held_out": best_unsupervised,
    "anomaly_hybrid_held_out": best_hybrid_record,
    "anomaly_synthetic_supervised_held_out": best_supervised_record,
    "checks": {
        "strict_forecast_chronology": True,
        "strict_anomaly_chronology": True,
        "past_only_behaviour_features": True,
        "training_only_scaling": True,
        "app_changed": False,
        "synthetic_supervised_result_is_diagnostic_only": True,
    },
}
results_path = OUTPUT_DIR / "spendly_optimized_results.json"
results_path.write_text(json.dumps(final_payload, indent=2, default=str), encoding="utf-8")

zip_path = Path("spendly_optimized_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))

print("SPENDLY_OPTIMIZED_RESULT_START")
print(json.dumps(final_payload, indent=2, default=str))
print("SPENDLY_OPTIMIZED_RESULT_END")
print("Saved:", zip_path.resolve())
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## Reading the result

- Prefer lower forecast MAE/WAPE and higher R², but inspect bias too.
- Treat the hybrid as the strongest immediately deployable anomaly experiment from this notebook.
- Treat the supervised Extra Trees result as a synthetic upper bound until real reviewed alerts exist.
- Do not change the application until these rerun results have been reviewed together.